In [ ]:
import json
import logging

import openeo
from openeo.rest.udp import build_process_dict

from utils import utils, urls, udp_params

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = udp_params.SPATIAL_EXTENT
temporal_extent = "2020-01-01"
canopy_cover_threshold = udp_params.CANOPY_COVER_THRESHOLD
natural_forest_threshold = udp_params.NATURAL_FOREST_THRESHOLD
min_connected_area = udp_params.MIN_CONNECTED_AREA

In [ ]:
parameters = [
    spatial_extent,
    canopy_cover_threshold,
    natural_forest_threshold,
    min_connected_area,
]

# UDP

In [ ]:
# Natural Forests of the World 2020
# EPSG:32636 = UTM zone 36N
# dims: ['x', 'y', 'bands']
natural_forest = connection.load_stac(
    url=urls.NATURAL_FOREST_STAC,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B0"],
)

In [ ]:
# remove the hidden time dimension
natural_forest = utils.drop_hidden_dimension(natural_forest, "t")

In [ ]:
# connection.describe_collection("ESA_WORLDCOVER_10M_2020_V1")

In [ ]:
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=ESA_WORLDCOVER_10M_2020_V1
# 10 m resolution
# EPSG:4326
# openEO backend: terrascope
esa_worldcover = connection.load_collection(
    "ESA_WORLDCOVER_10M_2020_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["MAP"],
)

In [ ]:
# remove the time dimension
esa_worldcover = esa_worldcover.drop_dimension("t")

In [ ]:
# connection.describe_collection("CLMS_TCD_PANTROPICAL_10M_YEARLY_V1")

In [ ]:
# tree cover density
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=CLMS_TCD_PANTROPICAL_10M_YEARLY_V1
# 10 m resolution
# EPSG:4326
# openEO backend: cdse
tree_cover_density = connection.load_collection(
    "CLMS_TCD_PANTROPICAL_10M_YEARLY_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["map"],
)

In [ ]:
# remove the time dimension
tree_cover_density = tree_cover_density.drop_dimension("t")

In [ ]:
# align all 3 datasets in UTM zone 36N
esa_worldcover = esa_worldcover.resample_cube_spatial(natural_forest, method="near")
tree_cover_density = tree_cover_density.resample_cube_spatial(
    natural_forest, method="near"
)

In [ ]:
# make sure band dimension has consistent labels
esa_worldcover = esa_worldcover.rename_labels(dimension="bands", target=["B0"])
tree_cover_density = tree_cover_density.rename_labels(dimension="bands", target=["B0"])

In [ ]:
# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gt_canopy_cover_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gt",
        "argument": canopy_cover_threshold,
    },
)

gt_natural_forest_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gt",
        "argument": natural_forest_threshold,
    },
)

In [ ]:
# mask, 1 = natural forest
forest_baseline = (
    (
        esa_worldcover == 10  # class 10 = tree cover
    )
    & (tree_cover_density.apply(gt_canopy_cover_threshold).convert_data_type("bool"))
    & (
        tree_cover_density <= 100  # values above 100 = unclassifiable / no_data
    )
    & (natural_forest.apply(gt_natural_forest_threshold).convert_data_type("bool"))
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "pixel_area": 10 * 10,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = forest_baseline.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [ ]:
inverse_small_region_mask = utils.invert_mask(small_region_mask)

In [ ]:
forest_baseline = forest_baseline & inverse_small_region_mask

# Serialise UDP

In [ ]:
summary = (
    "Construct a natural forest baseline pixel mask for the year 2020. "
    "The returned DataCube has a single band named B0. "
)
description = summary

udp_spec = build_process_dict(
    forest_baseline,
    process_id="forest_baseline",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A pixel mask. 1 = natural forest, 0 = non-forest.",
        "schema": {
            "type": "object",
            "subtype": "datacube"
        }
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)